## Course Announcements

Due Sunday: 
- A4
- Q5
- [mid-course survey](https://forms.gle/YTy2jZRsabW6fTHUA) (for EC)
- [weekly project survey](https://forms.gle/pRqEYm8rZQH4Ezcp7) (for EC) - complete weekly

Monday (expected): participation, A3, project proposal feedback

# Inference II: Relationships Between Variables

- Association between variables: **Pearson's correlation coefficient**
- Models: **Ordinary Least Squares regression (OLS)**
  - **simple linear regression**
    - effect size
    - p-value
    - interpretation
  - **multiple linear regression**
    - confounding
    - model selection

On your own (end of notebook): regression with transformed (log) data, categorical predictors, and multicollinearity.

## Contents

Lost track of where we are? Click a section to jump to it. (In JupyterLab, the table of contents sidebar, the bulleted-list icon on the left, shows the same outline and highlights the section you're in.)

- [1. Setup](#s1)
- [2. Correlation: Pearson's r](#s2) · [2a. No Correlation Example](#s2a)
- [3. The Data](#s3)
- [4. Simple Linear Regression](#s4) · [4a. Assumptions](#s4a) · [4b. Fitting the Model](#s4b) · [4c. Check the Model](#s4c) · [4d. Interpretation](#s4d) · [4e. Estimates](#s4e)
- [5. Multiple Linear Regression](#s5) · [5a. Outlier Handling](#s5a) · [5b. Before Fitting: Multicollinearity](#s5b) · [5c. Check the Model](#s5c) · [5d. Model Selection](#s5d)
- [6. On Your Own](#s6) · [6a. Regression with Transformed Data](#s6a) · [6b. Why Log-Transformation Isn't Trickery](#s6b) · [6c. Categorical Predictors](#s6c) · [6d. Multicollinearity](#s6d)

## 1. Setup <a id="s1"></a>

In [ ]:
import pandas as pd
import numpy as np

# Import seaborn and apply its plotting styles
import seaborn as sns
sns.set_theme(style="white", 
              font_scale=2, 
              rc={'axes.spines.right': False,'axes.spines.top': False}) # this last thing is equivalent to always having sns.despine()

# import matplotlib
import matplotlib.pyplot as plt
# set plotting size parameter
plt.rcParams['figure.figsize'] = (8, 8)

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

import warnings
warnings.filterwarnings('ignore')

#improve resolution
%config InlineBackend.figure_format ='retina'

## 2. Correlation: Pearson's $r$ <a id="s2"></a>

- Measures the strength of association between two variables 
- Takes values [-1,1]
- Positive Correlation 
    - $r$ > 0
    - as $r$ approaches 1, x and y are highly correlated
    - $r$ approaching 1 can happen with a shallow sloped line of regression
    - $r$ approaching (not quite) 0 can happen with a steep sloped line of regression
- Negative Correlation 
    - $r$ < 0
    - as $r$ approaches -1, x and y are highly negatively correlated
    - $r$ approaching -1 can happen with a shallow sloped line of regression
    - $r$ approaching (not quite) 0 can happen with a steep sloped line of regression
- Zero Correlation
   - $r \approx 0$
   - as $r$ approaches 0, x and y are not correlated
   - as $r$ approaches 0, the slope of the line of regression becomes 0 (horizontal)

Want to build your intuition? Try https://www.guessthecorrelation.com/

### 2a. No Correlation Example <a id="s2a"></a>


In [ ]:
np.random.seed(1)

# x and y are uncorrelated because they are independent draws of random numbers
# the value of x does not depend on the corresponding value of y and vice versa
x = np.random.randint(0, 50, 1000)
y = np.random.randint(0, 50, 1000)

plt.scatter(x, y, s=50);
print('correlation coefficients:\n', np.corrcoef(x, y))

#### Question #1

What would you expect Pearson correlation for the following relationship to be? Note that the value of y is a function of x plus some random noise added in.

- A) -0.8
- B) -0.4
- C) 0
- D) 0.4
- E) 0.8

https://forms.gle/vDUHuahSRQ39zfr59 

![](QRCodes/10_02.png)


In [ ]:
# setting a random seed makes this answer repeatable
np.random.seed(888)

# 1000 random integers between 0 and 50
x = np.random.randint(0, 50, 200)
y = x + np.random.normal(0, 10, 200)

plt.scatter(x, y, s=50);

In [ ]:
# now calculate it here!
np.corrcoef(x, y)

## 3. The Data <a id="s3"></a>
Here is some data we will use to explore inference.  Both for regression models and t-tests.  

The rows show (for a particular point in time) values for US states

The columns show
- Percent of people with incomes low enough to be classed as living in poverty
- Birth rates for mothers who are in different age bands (per 1000 females in band)
- An overall teen birth rate (combining different age bands)
- A rate of violent crimes committed (per 1000 individuals)


In [ ]:
# read in file; specify that it is tab-separated file
df = ( pd.read_csv('data/teen_birth.txt', sep='\t')
      .set_index('Location') )
df.head()

In [ ]:
# On your own, you could use this cell to explore the data...

# Look at univariate shape and stats
# Look at relationships between variables... what do scatter plots look like? are the correlations between variables?

#### Question #2

Hypothesis: What do you think the relationship between Poverty Percentage and Teen Birth Rate is? 

- A) $\beta_1=0$ No relationship
- B) $\beta_1>0$ As Poverty Percentage increases so does teen birth rate
- C) $\beta_1<0$ As Poverty Percentage increases, teen birth rate decreases

https://forms.gle/vDUHuahSRQ39zfr59 

![](QRCodes/10_02.png)

## 4. Simple Linear Regression <a id="s4"></a>

### 4a. Assumptions of Linear Regression <a id="s4a"></a>

| Assumption | When can you check it? | How |
|---|---|---|
| Linear relationship | **before** fitting (and again after) | scatterplot of the predictor vs. the outcome |
| No multicollinearity | **before** fitting | how strongly the predictors relate to each other, e.g. VIF (only matters with 2+ predictors; see 5b) |
| Independent observations (no autocorrelation) | before fitting, from how the data were collected | broken by repeated measures of the same people, or by time series |
| Equal variance (homoscedasticity) | **after** fitting | residuals vs. fitted plot (4c) |
| Normally distributed residuals | **after** fitting | QQ plot of the residuals (4c) |

The last two are assumptions about the model's **residuals** (observed − predicted), not about the raw variables. So you can't check them until you've fit the model. (This is also why a skewed outcome variable doesn't automatically break the normality assumption: what has to be roughly normal is the residuals.)

So the workflow is: **look at the data → fit → check the residuals → *then* interpret.**

Here, multicollinearity isn't an issue because we have only one predictor, and our rows are states (no time order), so autocorrelation isn't a concern. Let's check linearity, and look for outliers, by eye:

In [ ]:
# check linearity (and look for outliers) by eye
sns.lmplot(x="PovPct", y="TeenBrth", data=df);

### 4b. Fitting the Model <a id="s4b"></a>

$$outcome = \beta_0 + \beta_1*predictor$$

First, let's be sure we're on the same page about what our outcome is. Here, we're intererested in whether **Poverty Percentage (predictor)** impacts **Teen Birth Rate (outcome)**.

In [ ]:
model = smf.ols(formula='TeenBrth ~ PovPct', data=df)

In [ ]:
# model. # tab complete to see all the things inside the model object

We then fit the model and look at the results:

In [ ]:
## fit the model
results = model.fit()

## look at the results
print(results.summary())

#### Question #3

What is the effect size of the relationship between Poverty Percentage and Teen Birth Rate?

- A) 15.67
- B) 2.03
- C) 4.032
- D) 0.495

https://forms.gle/vDUHuahSRQ39zfr59 

![](QRCodes/10_02.png)

There is a lot of information in there. Let's focus on the three pieces from the slides:

- `coef` : $\beta_1$ estimate explaining the effect size
- `std err` : standard error 
- `P>|t|` : the p-value

### 4c. Check the Model <a id="s4c"></a>

Before we interpret anything in that summary, let's check the two assumptions we couldn't check before fitting: **equal variance** and **normally distributed residuals**. (We'll also take a second look at linearity.)

`statsmodels` has these tools built in: https://www.statsmodels.org/stable/diagnostic.html

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor

#### Residuals vs. fitted values: linearity & equal variance

Plot each state's residual against its fitted (predicted) value. If the assumptions hold, you'll see a shapeless cloud centered on 0, with about the same spread from left to right. The red line is a smoothed trend (LOWESS) to help your eye.

In [ ]:
sns.regplot(x=results.fittedvalues, y=results.resid, lowess=True,
            scatter_kws={'s': 80}, line_kws={'color': 'red'})
plt.axhline(0, color='gray', linestyle='--')
plt.xlabel('Fitted values (predicted Teen Birth Rate)')
plt.ylabel('Residuals');

What to look for:
- a **curve** means the relationship isn't linear: try transforming a variable (see 6a, in On Your Own) or adding a term like $x^2$
- a **funnel** (spread growing or shrinking from left to right) means unequal variance

The Breusch-Pagan test puts a number on the funnel check. Its null hypothesis is that the residuals have constant variance, so a **small** p-value is the bad result.

In [ ]:
bp_stat, bp_p, _, _ = het_breuschpagan(results.resid, results.model.exog)
print(f'Breusch-Pagan p = {bp_p:.3f}')

No evidence of unequal variance. (If there were, one fix is to keep the model but use standard errors that are robust to it: `smf.ols(...).fit(cov_type='HC3')`.)

#### Normality of the residuals: QQ plot

Same idea as the QQ plots in Inference I, but now applied to the residuals.

In [ ]:
sm.qqplot(results.resid, line='s')
plt.title('QQ plot of residuals');

jb_stat, jb_p, skew, kurtosis = jarque_bera(results.resid)
print(f'Jarque-Bera p = {jb_p:.3f}   (skew = {skew:.2f}, kurtosis = {kurtosis:.2f}; a normal distribution has 0 and 3)')

The points follow the line closely (the tails are a little short, which is not a concern), and Jarque-Bera shows no evidence of non-normal residuals. These numbers are also in the bottom panel of `results.summary()` (`Prob(JB)`, `Skew`, `Kurtosis`).

The assumptions look reasonable, so now we can trust (and interpret) the results.

### 4d. Interpretation <a id="s4d"></a>

$$outcome = \beta_0 + \beta_1*predictor$$

$$ Teen Birth = 15.67 + 2.03  * Poverty Percentage $$

If the Poverty Percentage were 0, the Teen Birth Rate would be **15.67** (The Intercept, $\beta_0$)

For every 1 unit increase in Poverty Percentage, you expect to see a **2.03** unit increase in Teen Birth Rate (The effect size, $\beta_1$)

In [ ]:
## look at the results
print(results.summary())

#### Question #4

Which value represents the expected Teen Birth Rate if the Poverty Percentage were 0?

- A) 15.67
- B) 2.03
- C) 4.032
- D) 0.495

https://forms.gle/vDUHuahSRQ39zfr59 

![](QRCodes/10_02.png)

In [ ]:
# Plot the orginal data (as before)
sns.scatterplot(x='PovPct', y='TeenBrth', data=df, s=100)

# Generate and plot the model fit line
xs = np.arange(df['PovPct'].min(), df['PovPct'].max())
ys = results.params.iloc[1] * xs + results.params.iloc[0]
plt.plot(xs, ys, '--k', linewidth=4, label='Model')

# make the title tell us model parameters
vals='$y={:.3f} + {:.3f}x, p={:.3f}, r={:.3f}, r^2={:.3f}$'.format(results.params.iloc[0], 
                                                                   results.params.iloc[1], 
                                                                   results.pvalues.iloc[1], 
                                                                   df[['PovPct','TeenBrth']].corr().iloc[0,1], 
                                                                   results.rsquared)
plt.title(vals);

plt.xlabel('Poverty Percentage')
plt.ylabel('Teen Birth Rate')
plt.legend();

The model (the line) mathematically describes the relationship between the data points, but it doesn't explain the relationship *perfectly*. (All models are wrong!)

Note that this line of regression comes from minimizing the Residual Sum of Squares (RSS):
- if you were to draw a vertical line from each point to the line and calculate that distance, that is the residual (OLS residuals are vertical: the difference between the observed and predicted *y*)
- The RSS is if you square that residual and then sum across all points
- this line of regression is the slope/intercept that minimizes RSS.
- the method of minimizing RSS is called Ordinary Least Squares or OLS

### 4e. Estimates <a id="s4e"></a>

If I told you a locations' Poverty Percentage, what would you guess its Teen Birth Rate would be?

https://forms.gle/vDUHuahSRQ39zfr59 

![](QRCodes/10_02.png)

#### Question #5

If I told you the Poverty Percentage of a state was 15, what would you estimate would be its Teen Birth Rate? 
- A) ~ 0
- B) ~ 30
- C) ~ 40
- D) ~ 50
- E) ~ 60

#### Question #6

Which of the following is a reasonable estimate (guess) for a state with a Poverty Percentage of 20?

- A) 55.99
- B) 56.27
- C) 56.5
- D) A - C
- E) None of the above

#### Question #7

What is our conclusion from this analysis? (Question: Does Poverty Percentage affect Teen Birth Rate?)

- A) Reject the null; There is no relationship between Poverty Percentage and Teen Birth Rate
- B) Reject the null; There is a relationship between Poverty Percentage and Teen Birth Rate
- C) Fail to reject the null; There is no relationship between Poverty Percentage and Teen Birth Rate
- D) Fail to reject the null; There is a relationship between Poverty Percentage and Teen Birth Rate


## 5. Multiple Linear Regression <a id="s5"></a>

So far, we've only been looking at the relationship of one measure (Poverty Percentage) on Teen Birth. 

*But*, Poverty Percentage is likely *not* the only thing that affects Teen Birth Rate.

We could imagine that Violent Crime rates in a location may affect both Poverty Percentage and could possibly affect Teen Birth Rate. (A confounder!)

This is where **multiple linear regression** is incredibly helpful. Multiple linear regression allows you to measure the effect of multiple predictors on an outcome.

Let's do some EDA first to see if our hypothesis (Violent Crime is related to Teen Birth) holds up in our data at all

In [ ]:
# relationship between predictors
sns.scatterplot(x='PovPct', y='ViolCrime', data=df, s=100)
plt.xlabel('Poverty Percentage')
plt.ylabel('Violent Crime');

### 5a. Outlier Handling <a id="s5a"></a>

Ugh! there's some nasty outlier going on there.... if we remove it maybe we will be able to see some relationship but we can't right now because that one huge violent crime location is messing up the plot

In [ ]:
df[df['ViolCrime'] > 50]

In [ ]:
# removing DC
df = df[df['ViolCrime'] < 50]

In [ ]:
# relationship between predictors
# after removing outlier
sns.scatterplot(x='PovPct', y='ViolCrime', data=df, s=100)
plt.xlabel('Poverty Percentage')
plt.ylabel('Violent Crime');

Yay, now we can see what looks like a potentially linear relationship between poverty and violent crime.

But BEWARE!  Removing an outlier WILL change the coefficients of the regression!

We did it here because the outlier was so big it might destroy the linear relationship.  But that's a judgement call, there is no "right way" when it comes to outlier removal, but there has to be a reason other than simply "it's an outlier".

Maybe if you've got time, you should rerun the analysis below WITHOUT removing DC... see what's different :)

### 5b. Before Fitting: Multicollinearity <a id="s5b"></a>

Now that we have two predictors, check that they aren't too strongly related to each other. This only uses the predictors, so we can do it before fitting.

The **variance inflation factor (VIF)** asks how well each predictor can be predicted from the other predictors. A VIF of 1 means not at all; above about 5-10 is a warning sign. (See 6d for what goes wrong when it's high.)

In [ ]:
predictors = sm.add_constant(df[['PovPct', 'ViolCrime']])
pd.Series([variance_inflation_factor(predictors.values, i) for i in range(1, predictors.shape[1])],
          index=predictors.columns[1:], name='VIF')

Both are about 1.4. Poverty and violent crime are correlated (r ≈ 0.5), but nowhere near enough to cause problems. Let's fit some models.

In [ ]:
mod = smf.ols(formula='TeenBrth ~ ViolCrime', data=df)
resvc = mod.fit()
print(resvc.summary())

In [ ]:

mod = smf.ols(formula='TeenBrth ~ PovPct + ViolCrime', data=df)
res = mod.fit()
print(res.summary())

### 5c. Check the Model <a id="s5c"></a>

Before interpreting the multiple regression, run the same residual checks as in 4c. With more than one predictor it's also worth checking whether any single observation is driving the fit, using **Cook's distance**: how much the whole model would change if you dropped that one observation. A common rule of thumb flags points above $4/n$.

Here's a function that does all of the after-fitting checks at once. Copy it into your project and pass it any fitted `statsmodels` OLS model.

In [ ]:
def regression_diagnostics(results):
    """Plots and tests for the after-fitting assumptions of a statsmodels OLS model."""
    resid, fitted = results.resid, results.fittedvalues
    n = int(results.nobs)

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 7))

    # 1. residuals vs. fitted: linearity & equal variance
    sns.regplot(x=fitted, y=resid, lowess=True, ax=ax1,
                scatter_kws={'s': 60}, line_kws={'color': 'red'})
    ax1.axhline(0, color='gray', linestyle='--')
    ax1.set(xlabel='Fitted values', ylabel='Residuals', title='Residuals vs. fitted')

    # 2. QQ plot: normality of residuals
    sm.qqplot(resid, line='s', ax=ax2)
    ax2.set_title('QQ plot of residuals')

    # 3. Cook's distance: influential observations
    cooks = results.get_influence().cooks_distance[0]
    ax3.stem(range(n), cooks)
    ax3.axhline(4 / n, color='red', linestyle='--')
    ax3.set(xlabel='Observation', ylabel="Cook's distance", title='Influence (red line = 4/n)')
    plt.tight_layout()
    plt.show()

    print(f"Breusch-Pagan p (equal variance):  {het_breuschpagan(resid, results.model.exog)[1]:.3f}")
    print(f"Jarque-Bera p (normal residuals):  {jarque_bera(resid)[1]:.3f}")
    # only meaningful if the rows have a natural order (e.g., time); ~2 = no autocorrelation
    print(f"Durbin-Watson (autocorrelation):   {durbin_watson(resid):.2f}")

regression_diagnostics(res)

No pattern in the residuals, the points follow the QQ line, and no single state stands far above the rest in Cook's distance. A few sit just above the $4/n$ line, and that's normal: the cutoff flags points to look at, not points to delete.

Remember DC? Here's the same model fit **with** DC included:

In [ ]:
df_all = pd.read_csv('data/teen_birth.txt', sep='\t').set_index('Location')
res_with_dc = smf.ols(formula='TeenBrth ~ PovPct + ViolCrime', data=df_all).fit()

cooks = pd.Series(res_with_dc.get_influence().cooks_distance[0], index=df_all.index)
print('rule-of-thumb cutoff (4/n):', round(4 / len(df_all), 3))
cooks.sort_values(ascending=False).head()

DC's Cook's distance is hundreds of times larger than any other state's. That's a number behind the decision we made in 5a, beyond "it looks weird on the plot." (It's still a judgement call, and one you should report.)

The checks look good, so let's interpret the model.

#### Question #8

https://forms.gle/vDUHuahSRQ39zfr59 

![](QRCodes/10_02.png)

Which has a larger effect on Teen Birth Rate?

- A) Poverty Percentage
- B) Violent Crime
- C) Effect is equal across all predictors
- D) No predictors have an effect on outcome


$$ Teen Birth = 15.3 + (1.19  * Poverty Percentage) + (1.63 * Violent Crime) $$

If the Poverty Percentage *and* Violent Crime were both 0, the Teen Birth Rate would be **15.3** (The Intercept, aka $\beta_0$)

Holding Violent crime constant, for every 1 unit increase in Poverty Percentage, you expect to see a **1.19** unit increase in Teen Birth Rate (The effect size, $\beta_1$)

Holding Poverty Percentage constant, for every 1 unit increase in Violent crime, you'd expect to see a **1.63** unit increase in Teen Birth Rate (The effect size, $\beta_2$)


### 5d. Model Selection <a id="s5d"></a>

We've now seen 3 different models
1. TB ~ PP
2. TB ~ VC
3. TB ~ PP + VC

Let's review the different model fits... hopefully they contain a way to select which model best fits the data! (spoiler, they do)

In [ ]:
from statsmodels.iolib.summary2 import summary_col

# fit all three models on the same 50 states (DC removed):
# model comparison metrics only make sense for models fit to the same data
models = {
    'TB ~ PP':      smf.ols(formula='TeenBrth ~ PovPct', data=df).fit(),
    'TB ~ VC':      smf.ols(formula='TeenBrth ~ ViolCrime', data=df).fit(),
    'TB ~ PP + VC': smf.ols(formula='TeenBrth ~ PovPct + ViolCrime', data=df).fit(),
}

# one table: coefficients (standard errors in parentheses), then fit statistics
summary_col(list(models.values()),
            model_names=list(models.keys()),
            regressor_order=['Intercept', 'PovPct', 'ViolCrime'],
            float_format='%.2f',
            info_dict={'AIC': lambda m: f'{m.aic:.1f}',
                       'BIC': lambda m: f'{m.bic:.1f}',
                       'N':   lambda m: f'{int(m.nobs)}'})

The bottom rows of this table (R-squared through BIC) are measures used for *in-sample* model selection.

In-sample means that these are measures meant to be used when you are evaluating how good a model is using the same data you are using to fit the model.  In-sample stands in contrast to out-of-sample methods you might have heard of like cross-validation (which uses different subsets of the data to fit a model than it does when selecting which model to use)

Lets start with R-squared.  R-squared is a **goodness-of-fit** measure for linear regression models. This statistic indicates the percentage of the variance in the dependent variable that the independent variables explain collectively.  We already know that Pearson's r having a value close to +1/-1 means the datapoints lie basically on a straight line.  When $r=1$the dots fall perfectly on the line, and knowing X uniquely determines Y... there is no noise.  When r is closer to zero noise is most of what we are seeing and knowing X doesn't tell us much about the corresponding Y value.  R-squared is literally Pearson's r, squared.  So it has the same properties plus it can be literally interpreted as "percent of the variance of Y which is explained by knowing X", AKA "percent variance explained". Models with high variance explained are models that fit their data well.

But selecting a model is MORE than just "did it fit the data well". 

To do in-sample model selection you want to ignore R-squared and instead look at ANY of the following metrics
- Adj. R-squared
- AIC
- BIC

Going into the math details of these different measures here is beyond the scope of this notebook.  For a full treatment I suggest Chapter 6 of the free textbook https://www.statlearning.com.  For a quick intro I found these slides useful https://pages.stat.wisc.edu/~ane/st572/notes/lec05.pdf

For an ever quicker introduction you need to understand an important principle.... the parsimony principle.  Parsimony is the idea (AKA, Occam's razor) that if two models fit the data equally well (e.g., have the roughly the same R-squared) then we should prefer the simpler model.  How is one model simpler than the others?  A simpler model has fewer X variables than a complex one.  If we try to fit a line through 10 datapoints in 2 dimensions then it has 2 parameters (slope/intercept) and that's pretty simple.  But if we try to fit a 9th order polynomial (it has terms with $x^9, x^8, \ldots, x$ plus an intercept) then we will *perfectly* hit all 10 datapoints with zero RSS.  But this wiggly curve will NOT generalize well to new data... we refer to this an overfit model because it is overly complex and fitting noise in the data as if it was signal.

Adjusted R-squared, AIC, and BIC are all metrics that are related to each other (again math details elsewhere).  These metrics reflect how good a model is at fitting data, but they also have complexity penalties built into them.  If two models fit the data the same, but one is more complex these metrics will prefer the simpler model.

Adjusted R-squared goes UP for better fit/simpler models and DOWN for worse fit/more complex models.

AIC and BIC do the opposite.  They go DOWN for better fit/simpler models and UP for worse fit/more complex models.

Any of these metrics can be used to select among different potential models (e.g., our 3 models above).  Generally BIC hates complex models more than the other two if you're looking to penalize model complexity.

So let's compare our 3 models using adjusted R-squared and BIC in the table above.... which model do you prefer?



Both measures pick the same model: `TB ~ PP + VC`. You'll use these to choose between models in A5.

---
## 6. On Your Own <a id="s6"></a>

We won't get to these in lecture, but they're worth working through before A5 and your project.

### 6a. Regression with Transformed Data <a id="s6a"></a>

So far, we've been working with data that were approximately Normal and didn't require transformation. But that won't always be the case...

In [ ]:
sleep = pd.read_csv('data/msleep.csv')
sleep.head()

What if we were interested in quantifying the **effect that REM sleep has on total sleep**?

We may hypothesize that the more REM sleep an animal gets the less total sleep it needs.

In [ ]:
sns.histplot(sleep['sleep_rem'].dropna())
plt.xlabel('REM Sleep (hours/night)');

These data are skewed right, and transforming these data could help us use them in analysis...

In [ ]:
sns.scatterplot(x='sleep_rem', y='sleep_total', data=sleep, s=100)
plt.xlabel('REM Sleep')
plt.ylabel('Total Sleep');

This relationship is not linear...so linear regression would not be appropriate

#### Log Transformation

In [ ]:
# add log-transformed column for sleep_rem
sleep['sleep_rem10'] = np.log10(sleep['sleep_rem'])

In [ ]:
# this now looks less skewed, more bell shaped
sns.histplot(sleep['sleep_rem10'].dropna())
plt.xlabel('log10(REM Sleep)');

In [ ]:
# and therefore, this now looks closer to a linear relationship
sns.scatterplot(x='sleep_rem10', y='sleep_total', data=sleep, s=100)
plt.xlabel('log10(REM Sleep)')
plt.ylabel('Total Sleep');

#### Linear Regression with Transformed Data

In [ ]:
# carry out regression using log-transformed predictor

mod = smf.ols(formula='sleep_total ~ sleep_rem10', data=sleep)
res = mod.fit()
print(res.summary())

In [ ]:
# view our log transformed lmplot
sns.lmplot(x='sleep_rem10',y='sleep_total',data=sleep);

In [ ]:
# compare above to the lmplot without log transformation
sns.lmplot(x='sleep_rem',y='sleep_total',data=sleep);

Here, we see that the $\beta_1$ estimate for `sleep_rem10` is 10.89....but remember that this value is on the log scale.

$$ Total Sleep = 8.57 + 10.89  * log_{10}(REM Sleep) $$

To interpret this coefficient, remember what the predictor is: $log_{10}$(REM Sleep).

So $\beta_1$ = 10.89 is the change in Total Sleep for a 1-unit increase in $log_{10}$(REM Sleep)... which is a **10x** increase in REM sleep. That's not a very useful comparison.

Instead, pick a change in REM sleep that *is* meaningful, and multiply $\beta_1$ by $log_{10}$ of that ratio:

In [ ]:
b1 = res.params['sleep_rem10']

print('REM sleep doubles:   Total Sleep changes by', round(b1 * np.log10(2), 2), 'hours')
print('REM sleep up 10%:    Total Sleep changes by', round(b1 * np.log10(1.1), 2), 'hours')

So, an animal that gets twice as much REM sleep is expected to sleep about **3.3 hours** more in total.

That's a sizable effect, and it's in the *opposite* direction from what we hypothesized at the beginning!

(You may have seen the shortcut "$\beta_1$/100 = change in the outcome for a **1% increase** in the predictor." That works for the natural log, `np.log()`, not for `np.log10()`. And it's for a *percent* change in the predictor, not a 1-unit change.)

More on interpretation of log transformed variables can be read [here](https://www.cscu.cornell.edu/news/statnews/stnews83.pdf)

### 6b. Why Is Log-Transformation Not Math Trickery? <a id="s6b"></a>

First, some reading on data transformation can be found [here](http://fmwww.bc.edu/repec/bocode/t/transint.html). It's not beautifully formatted, but I think it is written in a more-straightforward manner than some [other](https://stats.stackexchange.com/questions/4831/regression-transforming-variables/4833#4833) [places](https://stats.stackexchange.com/questions/298/in-linear-regression-when-is-it-appropriate-to-use-the-log-of-an-independent-va) [online](https://www.researchgate.net/post/Why_do_we_do_transformation_before_data_analysis).

But a basic understanding can be had in just a few bullet points
- measurement scales may be arbitrary. there's nothing to say log transforms aren't a better measurement scale than the original
- you are already familiar with many logrithmic measurement scales: decibels and Richter scale for measuring earthqueakes are two examples
- lots of things are power-law distributed or close to power-law distributed... the frequency of words in a language, human vision and auditory abilities in perception, the number of social media followers people have. 
- log transforms make a process that had been multiplicative or power law like ($a*b^n$) into an additive process ($a+n*b$) which really helps when you're trying to apply linear regression for obvious reasons
- log transforms can reduce heteroscadacity and skewness, which may make the data amenable to linear regression when it hadn't been before

The first thing to remember is there is (in most cases) nothing special about how the data are originally expressed.

In our example above, there is nothing about "hours" as the unit that was chosen that makes these data "correct".

So, while it _feels_ like data transformation is trickery, our initial unit of hours is...in some ways arbitrary and something that we chose.

This is where we'll start with our argument that it's ok to transform (or think of it as *re-express*) our data
so that it can be (still-accurately) used with well-studied models.

To make the point a little more concretely, all the following things are measured on a log scale: pH (measurement of acidity), dB (measure of sound loudness), and Richter scale (earthquake intensity). All of them _could_ be measured  on a linear scale. Those measurements would still explain those things...it would just be on a different scale.

In other words:
    
> "Transformations are needed because there is no guarantee that the world works on the scales it happens to be measured on."

What *does* differ however, is the interprtation. Linear scales tell us absolute change, while logarithmic scales tell us relative change.

### 6c. Regressing Categorical Variables <a id="s6c"></a>

In [ ]:
sns.histplot(df['ViolCrime'])
plt.xlabel('Violent Crime');

In [ ]:
df['ViolCrime'].describe()

In [ ]:
# add a new categorical column for a 'Violent' state, arbitrarily pick a threshold from histogram
df.loc[:, 'Violent'] = df['ViolCrime'].apply(lambda x: 0 if x < 11 else 1)
df.head()

In [ ]:
sns.boxplot(x='Violent', y='TeenBrth', data=df);

In [ ]:
mod = smf.ols(formula='TeenBrth ~ PovPct + C(Violent)', data=df)
res = mod.fit()
print(res.summary())

Notice: regressing on a two-level categorical variable is closely related to the t-test from Monday. Both ask whether the mean teen birth rate differs between the two groups.

### 6d. Multicollinearity <a id="s6d"></a>
Lets force a colinearity problem by making a synthetic variable to show you what to look out for...

specifically note that 
1. condition number is large... that's diagnosing the issue
2. y-intercept changes dramatically from before
3. the sum of coefficients doesn't arrive at the total it was at in previous model... again robbing it of explanatory power

In [ ]:
# generate artificially-colinear variable
df.loc[:, 'psuedoVC'] = df['ViolCrime'] * 3.6 + 112 + 5 * np.random.rand(len(df))

In [ ]:
# Plot relationship between colinear predictors (code added to notes)
sns.scatterplot(x='psuedoVC', y='ViolCrime', data=df, s=100);

In [ ]:
mod = smf.ols(formula='TeenBrth ~ PovPct + ViolCrime + psuedoVC', data=df)  
res = mod.fit()
print(res.summary())